# CW-ODMR

Plan for this notebook (building it up in stages):

1. **Resonator tuning (this section)** -- the resonator's tuning screw is manual, so this
   is an interactive loop: turn the screw a bit, take a quick NanoVNA sweep, look at where
   the S11 dip landed, repeat. No microwave source or laser needed for this part -- just
   the NanoVNA on the resonator's coupling port.
2. Once the resonator is close, do a coarse-then-fine sweep with the HP8673H + E4403B
   (`hp8673h.py`'s `resonance_sweep()`) to pin down the exact drive frequency and Q.
3. With the interlock running, power on the microwaves at that frequency, leave the laser
   on continuously, and record PL -- several long RTB2004 segments -- while sweeping (or
   sitting at) the MW frequency.

This first pass only covers step 1: taking data with the NanoVNA.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from nanovna import NanoVNAF3

vna = NanoVNAF3(debug=True)
print(vna.read_version())
print(vna.read_info())

NanoVNA-F V3: connected on COM6
version


0.6.0
info


Model:        NanoVNA-F_V3
Frequency:    1M ~ 6.3GHz
Build time:   Jun 18 2025 - 11:22:52 CST


## Resonator tuning loop

Fixed 2-4 GHz sweep range. Re-run the cell below after each turn of the tuning screw --
no need to reconnect or re-run the cells above.

In [ ]:
START_HZ = 2e9
STOP_HZ = 4e9
POINTS = 801  # NanoVNA-F V3's max

In [ ]:
# Re-run this cell after each screw adjustment.
freqs_hz, data = vna.sweep(START_HZ, STOP_HZ, POINTS, channels=(0,))
s11_db = 20 * np.log10(np.abs(data[0]))

dip_idx = np.argmin(s11_db)
dip_freq_hz = freqs_hz[dip_idx]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(freqs_hz / 1e9, s11_db)
ax.axvline(dip_freq_hz / 1e9, color="C1", ls="--", label=f"dip: {dip_freq_hz/1e9:.4f} GHz")
ax.set_xlabel("frequency (GHz)")
ax.set_ylabel("S11 (dB)")
ax.set_title("resonator reflection -- adjust the tuning screw and re-run")
ax.legend()
fig.tight_layout()

print(f"dip at {dip_freq_hz/1e9:.4f} GHz, depth {s11_db[dip_idx]:.1f} dB")

Once the dip sits at the target frequency (adjust `CENTER_HZ`/`SPAN_HZ` and re-run as it
gets close, to zoom in), save a final sweep of the tuned resonator for the record.

In [ ]:
import os
os.makedirs("data", exist_ok=True)
vna.run(START_HZ, STOP_HZ, POINTS, channels=(0,), path="data", name="resonator_tuned")

In [5]:
vna.close()